# Séance 1 — Explorer statistiquement les données avant le Machine Learning

## Fil rouge : le jeu de données `Diabetes`

### Objectifs

Dans cette première séance, on ne cherche **pas encore à construire le meilleur modèle**.

On commence par la question fondamentale :

> **Que contiennent nos données ?**

On va :
- charger un jeu de données ;
- identifier les variables et leur nature ;
- calculer des statistiques descriptives ;
- étudier les distributions ;
- détecter d'éventuelles valeurs atypiques ;
- étudier les associations entre variables ;
- comprendre la corrélation ;
- commencer à réfléchir aux relations entre les variables explicatives et la variable cible.

L'objectif est de prendre de bonnes habitudes d'analyse avant toute modélisation.

## 1. La démarche Machine Learning

Une chaîne classique peut être résumée ainsi :

**Données → Modèle → Prédictions → Fonction de perte → Optimisation → Paramètres**

Mais, en pratique, il faut ajouter une étape essentielle :

**Données → Exploration statistique → Préparation → Modèle → Évaluation**

Pourquoi ?

Parce qu'un modèle ne peut pas « comprendre » les données à notre place. Il faut notamment savoir :
- quelles sont les variables ;
- dans quelles unités elles sont exprimées ;
- si elles sont quantitatives ou qualitatives ;
- si elles sont corrélées ;
- s'il existe des valeurs manquantes ;
- si certaines variables ont une dispersion beaucoup plus importante que d'autres.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes

sns.set_theme()

## 2. Chargement du jeu de données

`scikit-learn` fournit plusieurs jeux de données pédagogiques.

Ici :

```python
load_diabetes()
```

renvoie un objet contenant notamment :
- `data` : les variables explicatives ;
- `target` : la variable à prédire ;
- `feature_names` : les noms des variables ;
- `DESCR` : une description du jeu de données.

In [ ]:
diabetes = load_diabetes()

print(type(diabetes))
print(diabetes.keys())

In [ ]:
print(diabetes.DESCR[:5000])

### Question d'interprétation

**À quoi correspond la variable `target` ?**

Prenez quelques instants pour lire la description fournie par `scikit-learn`.

## 3. Construire un DataFrame

Un DataFrame est une structure tabulaire très pratique pour l'analyse statistique.

Nous allons associer les noms des variables aux colonnes de `data`, puis ajouter `target`.

In [ ]:
df = pd.DataFrame(
    diabetes.data,
    columns=diabetes.feature_names
)

df["target"] = diabetes.target

df.head()

In [ ]:
print("Nombre d'observations :", df.shape[0])
print("Nombre de variables :", df.shape[1])

df.shape

In [ ]:
df.info()

### Variables explicatives et cible

On distingue :

- **features / variables explicatives** : `age`, `sex`, `bmi`, etc.
- **target / variable cible** : `target`

Dans notre problème :

$
X = (X_1,\ldots,X_p)
$

et nous cherchons à prédire :

$
Y = target
$

C'est donc un problème de **régression**, car `target` est quantitative.

## 4. Statistiques descriptives

La commande `describe()` donne notamment :
- le nombre d'observations ;
- la moyenne ;
- l'écart-type ;
- les quartiles ;
- le minimum ;
- le maximum.

In [ ]:
df.describe()

In [ ]:
df.describe().T

### Rappel statistique

Pour une variable quantitative :

**Moyenne**

$
\bar x = \frac{1}{n}\sum_{i=1}^n x_i
$

**Variance**

$
s^2 = \frac{1}{n-1}\sum_{i=1}^n(x_i-\bar x)^2
$

**Écart-type**

$
s = \sqrt{s^2}
$

L'écart-type mesure la dispersion autour de la moyenne.

In [ ]:
stats = pd.DataFrame({
    "moyenne": df.mean(),
    "mediane": df.median(),
    "ecart_type": df.std(),
    "variance": df.var(),
    "minimum": df.min(),
    "maximum": df.max()
})

stats

### Question d'interprétation

Comparez la moyenne et la médiane des variables.

- Pour quelles variables sont-elles proches ?
- Pour lesquelles sont-elles davantage éloignées ?
- Que peut suggérer un écart important entre moyenne et médiane ?

## 5. Valeurs manquantes

Avant de modéliser, il faut vérifier s'il existe des données manquantes.

In [ ]:
df.isna().sum()

In [ ]:
df.isna().sum().sum()

Ici, le jeu de données est propre : il n'y a pas de valeurs manquantes.

Dans un jeu de données réel, il faudrait décider comment traiter ces valeurs :
- suppression ;
- imputation par moyenne/médiane ;
- imputation plus sophistiquée ;
- modèle permettant de gérer les valeurs manquantes.

## 6. Distribution de la cible

Une première question importante est :

> **À quoi ressemble la distribution de ce que nous cherchons à prédire ?**

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["target"], kde=True)
plt.xlabel("target")
plt.title("Distribution de la variable cible")
plt.show()

In [ ]:
plt.figure(figsize=(8, 3))
sns.boxplot(x=df["target"])
plt.title("Boxplot de target")
plt.show()

### Rappel — boxplot

Un boxplot permet notamment de visualiser :
- la médiane ;
- les quartiles ;
- la dispersion ;
- certaines observations extrêmes.

Une valeur extrême n'est pas nécessairement une erreur.

> **Il ne faut jamais supprimer automatiquement une observation simplement parce qu'elle est atypique.**

In [ ]:
sns.violinplot(data=df["target"])

## 7. Distribution des variables explicatives

Regardons maintenant les variables explicatives.

In [ ]:
df.drop(columns="target").hist(
    figsize=(12, 10),
    bins=20
)

plt.suptitle("Distribution des variables explicatives")
plt.tight_layout()
plt.show()

### Une particularité du jeu Diabetes

Les variables explicatives de `load_diabetes()` ont été **centrées et mises à l'échelle**.

Cela explique pourquoi elles ont des valeurs assez petites.

C'est une situation particulière. Dans beaucoup de jeux de données réels, les variables peuvent avoir des unités et des échelles très différentes.

## 8. Les associations entre variables

Pour deux variables quantitatives, nous pouvons calculer leur corrélation.

La corrélation de Pearson est :

$
r_{XY}
=
\frac{Cov(X,Y)}
{\sigma_X\sigma_Y}
$

Elle est comprise entre -1 et 1.

- `r ≈ 1` : association linéaire positive forte ;
- `r ≈ -1` : association linéaire négative forte ;
- `r ≈ 0` : pas d'association linéaire importante.

⚠️ **Corrélation ne signifie pas causalité.**

In [ ]:
correlations = df.corr()

correlations

In [ ]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    correlations,
    annot=True,
    fmt=".2f",
    center=0
)

plt.title("Matrice de corrélation")
plt.show()

In [ ]:
correlations["target"].sort_values(ascending=False)

### Question d'interprétation

1. Quelle variable semble avoir l'association linéaire positive la plus forte avec `target` ?
2. Quelle variable semble avoir l'association négative la plus forte ?
3. Est-ce que la variable la plus corrélée explique nécessairement la majorité de la variabilité de `target` ?

## 9. Nuages de points

La corrélation donne un résumé numérique.

Le graphique permet de voir la relation directement.

In [ ]:
variables = ["bmi", "s5", "bp"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, variable in zip(axes, variables):
    sns.scatterplot(
        data=df,
        x=variable,
        y="target",
        ax=ax
    )
    ax.set_title(f"{variable} vs target")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, variable in zip(axes, variables):
    sns.regplot(
        data=df,
        x=variable,
        y="target",
        ax=ax
    )
    ax.set_title(f"Régression : {variable} vs target")

plt.tight_layout()
plt.show()

### Important

La droite représentée par `regplot` est une **droite descriptive**.

Nous n'avons pas encore construit notre modèle de régression multiple.

La prochaine séance consistera justement à formaliser cette idée.

## 10. Corrélation entre variables explicatives

Il faut aussi regarder si les variables explicatives sont elles-mêmes associées.

Cela sera important pour comprendre l'interprétation des coefficients d'une régression.

In [ ]:
features = diabetes.feature_names

corr_features = df[features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_features,
    annot=True,
    fmt=".2f",
    center=0
)
plt.title("Corrélations entre variables explicatives")
plt.show()

### Question finale

Pourquoi peut-il être intéressant de connaître les associations entre **variables explicatives** avant de construire une régression ?

# Bilan de la séance

Nous avons appris à :

- charger un jeu de données ;
- construire un DataFrame ;
- distinguer `X` et `y` ;
- calculer des statistiques descriptives ;
- étudier les distributions ;
- vérifier les valeurs manquantes ;
- calculer et visualiser des corrélations ;
- étudier graphiquement les associations.

### À retenir

> **L'analyse statistique des données est une étape du Machine Learning, pas une étape facultative.**

La prochaine séance : **régression linéaire → coefficients → fonction de perte → R² → significativité statistique**.